# Local Invoice OCR on Google Colab
Select **Runtime > Change runtime type > T4 GPU**, reconnect, then run every cell in order. By default the notebook stops if no GPU is attached. PDFs come from the public GitHub repo; no Drive upload is needed. Accuracy mode reads original PDF pages with a vision model, then checks item arithmetic and OCR evidence; Fast mode uses spatial OCR only. Stock Ollama remains the default. Set USE_TRAINED_ADAPTER=True and paste TRAINED_RELEASE_TAG only after colab_train.ipynb publishes a passing adapter to a public GitHub Release.

In [ ]:
#@title 1. Clone the public GitHub project
import os, pathlib, subprocess
PROJECT_DIR = pathlib.Path('/content/OCR')
if not PROJECT_DIR.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ubaid-148/OCR.git',str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR/'.git').is_dir():
    subprocess.run(['git','-C',str(PROJECT_DIR),'pull','--ff-only'], check=True)
else: raise ValueError('/content/OCR exists but is not a Git clone. Start a fresh runtime.')
os.chdir(PROJECT_DIR)
print('Project ready at', PROJECT_DIR)
print('Flow 2026-09-trained-gated-v13: approved adapter optional; uncertain fields stay in review')

In [ ]:
#@title 2. Install dependencies and verify the OCR device
import os, pathlib, shutil, subprocess, sys
REQUIRE_GPU_FOR_ACCURACY = True #@param {type:"boolean"}
gpu_runtime = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0
print('GPU attached:', gpu_runtime, flush=True)
if REQUIRE_GPU_FOR_ACCURACY and not gpu_runtime:
    raise RuntimeError('No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, then run all cells again. Accuracy vision on CPU caused a 180-second timeout and must not silently fall back.')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'tesseract-ocr-eng', 'tesseract-ocr-ara', 'tesseract-ocr-urd', 'ghostscript', 'unpaper', 'pngquant', 'zstd'], check=True)
# Keep OCR packages separate from Colab's preinstalled CUDA PyTorch.
OCR_ENV_DIR = pathlib.Path('/content/ocr-runtime')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(OCR_ENV_DIR)], check=True)
OCR_PYTHON = str(OCR_ENV_DIR / 'bin' / 'python')
# pip --python bootstraps pip even in a venv created without it.
ocr_pip = [sys.executable, '-m', 'pip', '--python', OCR_PYTHON]
subprocess.run([*ocr_pip, 'install', '-q', '--upgrade', 'pip'], check=True)
# ModelScope imports torch; its CPU build avoids a second CUDA/NCCL stack.
subprocess.run([*ocr_pip, 'install', '-q', 'torch==2.9.1+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
# CPU and GPU Paddle share a module: install exactly one distribution.
subprocess.run([*ocr_pip, 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=True)
requirements = [line.strip() for line in pathlib.Path('requirements.txt').read_text().splitlines() if line.strip() and not line.strip().startswith('paddlepaddle')]
subprocess.run([*ocr_pip, 'install', '-q', *requirements], check=True)
command = [*ocr_pip, 'install', '-q']
if gpu_runtime:
    command += ['paddlepaddle-gpu==3.3.1', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/']
else:
    command += ['paddlepaddle==3.3.1']
subprocess.run(command, check=True)
# Retry uncertain identifiers/numeric cells with bounded English OCR crops.
os.environ['OCR_TARGETED_RETRY'] = 'true'
os.environ['OCR_DEVICE'] = 'gpu:0' if gpu_runtime else 'cpu'
os.environ['VISION_REQUIRE_GPU'] = 'true'
os.environ.pop('OCR_PYTHON_EXE', None)
# Check in a fresh process so rerunning this cell cannot reuse an old Paddle import.
os.environ.setdefault('FLAGS_use_mkldnn', '0')
verification = subprocess.run(
    [OCR_PYTHON, '-u', str(PROJECT_DIR / 'check_ocr_runtime.py')],
    cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, errors='replace',
)
verification_log = pathlib.Path('/tmp/ocr-runtime-check.log')
verification_log.write_text(verification.stdout, encoding='utf-8')
print(verification.stdout, flush=True)
if verification.returncode:
    raise RuntimeError(
        f'OCR runtime verification failed (exit {verification.returncode}). '
        f'Full log: {verification_log}. Copy the error below:\n\n'
        + verification.stdout[-12000:]
    )
print('Dependencies ready. First upload loads OCR models; later uploads reuse them.')


In [ ]:
#@title 3. Start stock vision AI or an approved trained adapter
USE_LOCAL_AI = True #@param {type:"boolean"}
USE_TRAINED_ADAPTER = False #@param {type:"boolean"}
TRAINED_RELEASE_TAG = '' #@param {type:"string"}
OLLAMA_MODEL = 'qwen3-vl:4b' #@param {type:"string"}
import json, os, shutil, subprocess, sys, time, urllib.request
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from ollama_http import preload
if USE_TRAINED_ADAPTER: USE_LOCAL_AI = True
os.environ['OLLAMA_MODEL'] = OLLAMA_MODEL
os.environ['USE_LOCAL_AI'] = str(USE_LOCAL_AI).lower()
if USE_LOCAL_AI and not gpu_runtime:
    raise RuntimeError('Vision AI is disabled on CPU for this notebook. Attach a T4/L4 GPU, reconnect, and rerun from cell 1; or set USE_LOCAL_AI=False for spatial-only Fast mode.')
os.environ['OLLAMA_TIMEOUT_SECONDS'] = '180'
os.environ['OLLAMA_NUM_CTX'] = '16384'
os.environ['OLLAMA_NUM_PREDICT'] = '4096'
os.environ.pop('OLLAMA_URL', None)
os.environ.pop('TRAINED_VISION_URL', None)
if USE_TRAINED_ADAPTER:
    if not TRAINED_RELEASE_TAG: raise ValueError('Paste the release tag printed by colab_train.ipynb cell 11')
    from training.github_release import download_bundle
    approval_path = download_bundle(TRAINED_RELEASE_TAG, pathlib.Path('/content/approved-invoice-adapter')/TRAINED_RELEASE_TAG)
    from training.adapter_service import load_approval
    approved = load_approval(approval_path)
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.0','accelerate==1.7.0','peft==0.17.1','qwen-vl-utils==0.0.14','pillow'], check=True)
    subprocess.run(['pkill','-TERM','-x','ollama'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    if 'adapter_process' in globals() and adapter_process.poll() is None:
        adapter_process.terminate(); adapter_process.wait(timeout=15)
    adapter_log = open('/tmp/invoice-adapter.log','w')
    adapter_process = subprocess.Popen([sys.executable,'-u','-m','training.adapter_service','--approval',str(approval_path)], cwd=PROJECT_DIR, stdout=adapter_log, stderr=subprocess.STDOUT)
    for attempt in range(600):
        if adapter_process.poll() is not None:
            raise RuntimeError('Approved adapter service failed: '+pathlib.Path('/tmp/invoice-adapter.log').read_text()[-4000:])
        try:
            with urllib.request.urlopen('http://127.0.0.1:8766/health',timeout=2) as health:
                if json.load(health).get('ready'): break
        except OSError: time.sleep(1)
    else: raise RuntimeError('Approved adapter did not become ready; see /tmp/invoice-adapter.log')
    os.environ['TRAINED_VISION_URL'] = 'http://127.0.0.1:8766/extract'
    print('Approved trained adapter ready:', approved['model_id'])
elif USE_LOCAL_AI:
    if not shutil.which('ollama'):
        ollama_archive = '/tmp/ollama-linux-amd64.tar.zst'
        print('Downloading Ollama...')
        urllib.request.urlretrieve('https://ollama.com/download/ollama-linux-amd64.tar.zst', ollama_archive)
        subprocess.run(['tar', '--zstd', '-xf', ollama_archive, '-C', '/usr'], check=True)
        if not shutil.which('ollama'):
            raise RuntimeError('Ollama archive extracted but the executable was not found')
    # Cell 1 replaces /content/OCR. Restart Ollama so it never retains that deleted cwd.
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2) as response:
            service_running = response.status == 200
    except OSError:
        service_running = False
    if service_running:
        subprocess.run(['ollama', 'stop', OLLAMA_MODEL], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        subprocess.run(['pkill', '-TERM', '-x', 'ollama'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        time.sleep(2)
    if 'ollama_log' in globals() and not ollama_log.closed:
        ollama_log.close()
    ollama_log = open('/tmp/ollama.log', 'a')
    ollama_process = subprocess.Popen(
        ['ollama', 'serve'], cwd='/content', start_new_session=True,
        stdout=ollama_log, stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama did not start. Check /tmp/ollama.log')
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    print('Loading AI model before the first invoice...')
    ai_preloaded = preload(OLLAMA_MODEL, context=int(os.environ['OLLAMA_NUM_CTX']))
    if not ai_preloaded:
        raise RuntimeError('Vision model did not preload. Check /tmp/ollama.log; the app will not launch into a slow spatial fallback.')
    placement = subprocess.run(['ollama', 'ps'], capture_output=True, text=True, check=True)
    print(placement.stdout, flush=True)
    model_line = next((line for line in placement.stdout.splitlines()[1:] if line.split() and line.split()[0] == OLLAMA_MODEL), '')
    if '100% GPU' not in model_line:
        raise RuntimeError(f'{OLLAMA_MODEL} is not fully on GPU. Ollama placement: {model_line or placement.stdout}. Use a runtime with more free GPU memory; CPU offload is too slow for Accuracy mode.')
    print('Ollama setup complete: vision model preloaded fully on GPU.')
else:
    os.environ['OLLAMA_URL'] = 'http://127.0.0.1:1/api/chat'
    print('Local AI disabled; deterministic spatial fallback will be used.')

In [ ]:
#@title 4. Start OCR web application
import os, subprocess, sys, time, urllib.request
os.chdir('/content/OCR')
if 'OCR_PYTHON' not in globals():
    raise RuntimeError('Run dependency setup (cell 2) first.')
if 'ocr_process' in globals() and ocr_process.poll() is None:
    ocr_process.terminate()
    ocr_process.wait(timeout=15)
os.environ['OCR_PRELOAD'] = 'true'
print('Preparing OCR models before accepting uploads; first startup may download models.')
ocr_log = open('/tmp/ocr-web.log', 'w')
ocr_process = subprocess.Popen([OCR_PYTHON, '-u', 'ocr_web.py'], stdout=ocr_log, stderr=subprocess.STDOUT, env=os.environ.copy())
for attempt in range(600):
    if attempt and attempt % 15 == 0:
        print('Still preparing OCR models. Recent log:', open('/tmp/ocr-web.log').read()[-600:])
    if ocr_process.poll() is not None:
        print(open('/tmp/ocr-web.log').read())
        raise RuntimeError('OCR server exited during startup')
    try:
        response = urllib.request.urlopen('http://127.0.0.1:8765/', timeout=2)
        if response.status == 200:
            break
    except Exception:
        time.sleep(1)
else:
    print(open('/tmp/ocr-web.log').read())
    raise RuntimeError('OCR web application did not start')
print('OCR application is ready.')

In [ ]:
#@title 5. Open the application
from google.colab import output
output.serve_kernel_port_as_iframe(8765, height='700')

In [ ]:
#@title 6. One-time 9498 source regression check (not model training)
RUN_9498_CHECK = True #@param {type:"boolean"}
if RUN_9498_CHECK and USE_LOCAL_AI:
    import json, pathlib, urllib.request, uuid
    sample_path = PROJECT_DIR / 'public_invoice_pdfs' / '9498.pdf'
    boundary = 'OCRQA' + uuid.uuid4().hex
    fields = [('languages', b'eng+ara'), ('mode', b'auto'), ('format', b'invoice')]
    body = bytearray()
    for name, value in fields:
        body += f'--{boundary}\r\nContent-Disposition: form-data; name=\"{name}\"\r\n\r\n'.encode() + value + b'\r\n'
    body += f'--{boundary}\r\nContent-Disposition: form-data; name=\"pdf\"; filename=\"9498.pdf\"\r\nContent-Type: application/pdf\r\n\r\n'.encode()
    body += sample_path.read_bytes() + b'\r\n' + f'--{boundary}--\r\n'.encode()
    request = urllib.request.Request('http://127.0.0.1:8765/', data=bytes(body), headers={'Content-Type': f'multipart/form-data; boundary={boundary}'})
    print('Checking 9498.pdf with the running Colab app; this can take time on first model load...')
    with urllib.request.urlopen(request, timeout=900) as response:
        qa_result = json.load(response)
    qa_data = qa_result.get('data') or {}
    qa_items = qa_data.get('items') or []
    qa_checks = {
        'parser_financially_complete': qa_result.get('parser') in ('visual_ai','trained_visual_ai'),
        'invoice_number': (qa_data.get('invoice') or {}).get('invoice_number') == '2690111862',
        'invoice_date': (qa_data.get('invoice') or {}).get('date') == '2026-03-09',
        'supply_date': (qa_data.get('invoice') or {}).get('date_of_supply') == '2026-04-02',
        'seller_vat': (qa_data.get('supplier') or {}).get('vat_number') == '310981818100003',
        'seller_english_name': (qa_data.get('supplier') or {}).get('name_en') == 'Abdulrahman Ahmed Alrajhi Trading Co.',
        'customer_vat': (qa_data.get('customer') or {}).get('vat_number') == '300402905100003',
        'customer_building': (qa_data.get('customer') or {}).get('building_no') == '3518',
        'customer_address_not_seller': bool((qa_data.get('customer') or {}).get('address')) and '6595' not in str((qa_data.get('customer') or {}).get('address')),
        'item_codes': [row.get('item_code') for row in qa_items] == ['1212', '1218', '5007'],
        'item_quantities': [row.get('quantity') for row in qa_items] == [2, 1, 1],
        'item_descriptions': len(qa_items) == 3 and 'HELIX' in str(qa_items[0].get('description') or '').upper() and '15/40' in str(qa_items[0].get('description') or '') and 'HELIX' in str(qa_items[1].get('description') or '').upper() and 'D4' in str(qa_items[2].get('description') or '').upper(),
        'item_prices': [row.get('unit_price') for row in qa_items] == [14.79, 50.44, 14.79],
        'item_amounts': [row.get('amount') for row in qa_items] == [14.79, 50.44, 14.79],
        'item_vat': [row.get('vat_amount') for row in qa_items] == [2.22, 7.57, 2.22],
        'item_gross': [row.get('gross_amount') for row in qa_items] == [34.02, 58.01, 17.01],
        'totals': [(qa_data.get('totals') or {}).get(key) for key in ('subtotal', 'vat_amount', 'net_amount')] == [94.81, 14.22, 109.03],
    }
    vision_executed = qa_result.get('local_ai_status') in ('vision_evidence_reviewed', 'trained_evidence_reviewed', 'incomplete_reconciled', 'rejected_unsafe_vision_result')
    regression_passed = all(qa_checks.values())
    if not regression_passed:
        print('FAIL: 9498 is NOT accurate enough for automatic accounting import. Review failed fields against the PDF.')
    print(json.dumps({'vision_executed': vision_executed, 'regression_passed': regression_passed, 'observed_passed': sum(qa_checks.values()), 'total': len(qa_checks), 'checks': qa_checks, 'ocr_device': qa_result.get('ocr_device'), 'parser': qa_result.get('parser'), 'local_ai_error': qa_result.get('local_ai_error'), 'timings_seconds': qa_result.get('timings_seconds')}, indent=2))
    print('Full extraction:')
    print(json.dumps(qa_result, ensure_ascii=False, indent=2))
elif RUN_9498_CHECK:
    print('9498 vision check skipped because USE_LOCAL_AI=False.')
else:
    print('One-time 9498 check disabled.')

Upload any PDF in the embedded application. Default Accuracy mode uses stock Ollama and is **not trained** on these PDFs. Optional USE_TRAINED_ADAPTER downloads a public GitHub Release only after the held-out gate passes. Both paths reconcile with positioned OCR and keep uncertain results as needs_review; neither guarantees Cloud-level accuracy. Cell 6 reports regression_passed only when all known 9498 checks pass. Select Fast for spatial OCR only. Colab processes are temporary; rerun from cell 1 after updates. If startup fails, inspect `!/tmp/ocr-web.log`, `!/tmp/ollama.log`, or `!/tmp/invoice-adapter.log`.